In [ ]:
import pandas as pd
import numpy as np
import warnings
import pickle
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')

# ==========================================
# 1. LOAD DATA
# ==========================================
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Initial Train Shape: {train.shape}")
print(f"Initial Test Shape: {test.shape}")

# Save IDs for submission
test_id = test['Id']

# Remove Outliers (Crucial for Ames Housing Dataset)
train = train.drop(train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index)

# Log Transform Target Variable (RMSLE Alignment)
y_train = np.log1p(train['SalePrice'])

# Drop ID and Target
train_features = train.drop(['Id', 'SalePrice'], axis=1)
test_features = test.drop(['Id'], axis=1)

# Combine for uniform processing
all_data = pd.concat([train_features, test_features], axis=0).reset_index(drop=True)

# ==========================================
# 2. DOMAIN-SPECIFIC MISSING VALUE IMPUTATION
# ==========================================
# Categorical features where NaN means "None / Not Present"
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]
for col in none_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna('None')

# Numerical features where NaN means 0
zero_cols = [
    'GarageArea', 'GarageCars', 'GarageYrBlt',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
    'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
]
for col in zero_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(0)

# Impute LotFrontage with Median of the Neighborhood
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# Impute remaining missing values with Mode / Median
all_data['MSZoning'] = all_data['MSZoning'].fillna(all_data['MSZoning'].mode()[0])
all_data['Electrical'] = all_data['Electrical'].fillna(all_data['Electrical'].mode()[0])
all_data['KitchenQual'] = all_data['KitchenQual'].fillna(all_data['KitchenQual'].mode()[0])
all_data['Exterior1st'] = all_data['Exterior1st'].fillna(all_data['Exterior1st'].mode()[0])
all_data['Exterior2nd'] = all_data['Exterior2nd'].fillna(all_data['Exterior2nd'].mode()[0])
all_data['SaleType'] = all_data['SaleType'].fillna(all_data['SaleType'].mode()[0])
all_data['Functional'] = all_data['Functional'].fillna('Typ')
all_data['Utilities'] = all_data['Utilities'].fillna('AllPub')

# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================
# Total Square Feet
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

# Total Bathrooms
all_data['TotalBath'] = (all_data['FullBath'] + (0.5 * all_data['HalfBath']) +
                         all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath']))

# Age features
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['IsRemodeled'] = (all_data['YearBuilt'] != all_data['YearRemodAdd']).astype(int)
all_data['IsNew'] = (all_data['YrSold'] == all_data['YearBuilt']).astype(int)

# Overall Score Combination
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']

# Total Porch Area
all_data['TotalPorchSF'] = (all_data['OpenPorchSF'] + all_data['3SsnPorch'] +
                            all_data['EnclosedPorch'] + all_data['ScreenPorch'] + 
                            all_data['WoodDeckSF'])

# Convert MSSubClass to string (it is categorical, not numerical)
all_data['MSSubClass'] = all_data['MSSubClass'].astype(str)

# ==========================================
# 4. ONE-HOT ENCODING & DATA SPLITTING
# ==========================================
all_data = pd.get_dummies(all_data, drop_first=True)

# Split back to Train and Test sets
n_train = len(y_train)
X_train = all_data.iloc[:n_train, :]
X_test = all_data.iloc[n_train:, :]

print(f"Processed Train Shape: {X_train.shape}")
print(f"Processed Test Shape: {X_test.shape}")

# Scale features for Linear/Ridge Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 5. MODEL TRAINING & CROSS-VALIDATION
# ==========================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_rmse(model, X, y):
    rmse = np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=kf))
    return rmse

# Model 1: Optimized XGBoost Regressor
model_xgb = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=3,
    min_child_weight=3,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

# Model 2: LightGBM Regressor
model_lgb = lgb.LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.01,
    max_depth=3,
    num_leaves=15,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

# Model 3: Ridge Regression
model_ridge = Ridge(alpha=15.0, random_state=42)

# Evaluate Models
score_xgb = cv_rmse(model_xgb, X_train, y_train)
print(f"\nXGBoost CV RMSLE Score: {score_xgb.mean():.4f} (+/- {score_xgb.std():.4f})")

score_lgb = cv_rmse(model_lgb, X_train, y_train)
print(f"LightGBM CV RMSLE Score: {score_lgb.mean():.4f} (+/- {score_lgb.std():.4f})")

score_ridge = cv_rmse(model_ridge, X_train_scaled, y_train)
print(f"Ridge CV RMSLE Score:    {score_ridge.mean():.4f} (+/- {score_ridge.std():.4f})")

# ==========================================
# 6. BLENDING PREDICTIONS & SUBMISSION
# ==========================================
# Fit models on full train data
model_xgb.fit(X_train, y_train)
model_lgb.fit(X_train, y_train)
model_ridge.fit(X_train_scaled, y_train)

# Save XGBoost model
with open("xgb_model_optimized.pkl", "wb") as f:
    pickle.dump(model_xgb, f)

# Generate predictions (in log scale)
pred_xgb = model_xgb.predict(X_test)
pred_lgb = model_lgb.predict(X_test)
pred_ridge = model_ridge.predict(X_test_scaled)

# Weighted Average Ensemble Blending
final_log_preds = (0.45 * pred_xgb) + (0.35 * pred_lgb) + (0.20 * pred_ridge)

# Convert back from Log Scale to Actual SalePrice
final_preds = np.expm1(final_log_preds)

# Create Submission File
submission = pd.DataFrame({
    'Id': test_id,
    'SalePrice': final_preds
})

submission.to_csv('submission_optimized_blend.csv', index=False)
print("\nSaved predictions to 'submission_optimized_blend.csv'")